In [1]:
import numpy as np
from matplotlib import pyplot as plt
import pickle

In [2]:
import pickle
import numpy as np
import pandas as pd

def get_omega_norm_mc(_model, MC):
    self = _model.observation_layer.RFF
    eps = self.normal_generator.sample_univariate(0, 1, (MC, self.Dx, self.J))
    omega_norm = self.mu + np.exp(self.log_sigma) * eps
    return omega_norm # (MC, D, J)

# Define parameter ranges
J_vals = [8, 16, 32, 64, 128]
seeds = [1, 5, 12, 14, 16]
model_names = ['i.i.d. RBF', 'i.i.d. Maxwell', 'H-DPP']

# Initialize a dictionary to store rows for the final DataFrame
table_data = []

for J in J_vals:
    # Initialize containers to collect errors across all seeds for the current J
    errors_dict = {'i.i.d. RBF': [], 'i.i.d. Maxwell': [], 'H-DPP': []}

    for model_seed in seeds:
        try:
            # Load models for the current J and seed
            with open(f'concrete_maxwell_{J}_{model_seed}.pkl', 'rb') as f:
                max_model = pickle.load(f)
            with open(f'concrete_orf_{J}_{model_seed}.pkl', 'rb') as f:
                orf_model = pickle.load(f)
            with open(f'concrete_rbf_{J}_{model_seed}.pkl', 'rb') as f:
                rbf_model = pickle.load(f)

            # Extract W matrices
            W_max = np.average(get_omega_norm_mc(max_model, 10), axis=0)
            W_hdd = np.average(get_omega_norm_mc(orf_model, 10), axis=0)
            W_rbf = np.average(get_omega_norm_mc(rbf_model, 10), axis=0)

            # Calculate Frobenius norm error for each model
            for W, name in zip([W_rbf, W_max, W_hdd], model_names):
                error = np.linalg.norm(W.T @ W - np.eye(J), ord='fro')
                errors_dict[name].append(error)

        except FileNotFoundError:
            # Handle missing files gracefully if any seed/J combination is absent
            print(f"Warning: File not found for J={J}, seed={model_seed}. Skipping.")
            continue

    # Prepare the row data for the current J
    row_data = {'J': J}
    for name in model_names:
        if errors_dict[name]:
            mean_err = np.mean(errors_dict[name])
            std_err = np.std(errors_dict[name])
            # Format as "mean ± std"
            row_data[name] = f"{mean_err:.4f} ± {std_err:.4f}"
        else:
            row_data[name] = "N/A"

    table_data.append(row_data)

# Create DataFrame and set 'J' as the index
df_results = pd.DataFrame(table_data)
df_results.set_index('J', inplace=True)

# Display the final table
print("\n--- Approximation Error (Frobenius Norm: Mean ± Std) ---")
print(df_results.to_string())


--- Approximation Error (Frobenius Norm: Mean ± Std) ---
                i.i.d. RBF    i.i.d. Maxwell             H-DPP
J                                                             
8         29.9114 ± 7.1702   2.9845 ± 0.4339   2.8370 ± 0.1316
16        50.8868 ± 1.9861   6.9234 ± 0.5423   6.7873 ± 1.0178
32   1448.9511 ± 1942.9541  13.2741 ± 1.2542  12.0242 ± 0.4202
64     309.7710 ± 201.5668  25.2194 ± 0.4873  24.1718 ± 0.3986
128      381.2566 ± 2.6307  47.6876 ± 0.6079  45.9009 ± 0.2965
